In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# === đường dẫn ===
raw_prices_path = Path('data/raw/prices_adjclose.parquet')
proc_dir = Path('data/processed')
proc_dir.mkdir(parents=True, exist_ok=True)

out_calendar   = proc_dir / 'calendar.parquet'
out_prices_aln = proc_dir / 'prices_aligned.parquet'
out_ret_lin    = proc_dir / 'returns_linear.parquet'
out_ret_log    = proc_dir / 'returns_log.parquet'
out_bench_ret  = proc_dir / 'benchmark_ret.parquet'

# === đọc giá gốc ===
prices = pd.read_parquet(raw_prices_path)
prices.index = pd.to_datetime(prices.index)

# === tạo lịch giao dịch đơn giản (business days) ===
start_date = prices.index.min().strftime('%Y-%m-%d')
end_date   = prices.index.max().strftime('%Y-%m-%d')
cal = pd.date_range(start=start_date, end=end_date, freq='B')  # business day generic
calendar = pd.DataFrame({'date': cal})
calendar.to_parquet(out_calendar, index=False)

# === align giá theo calendar và ffill ngày nghỉ khác nhau ===
prices_aligned = prices.reindex(cal)
prices_aligned = prices_aligned.ffill()  # giữ nguyên bản chất Adj Close
prices_aligned.to_parquet(out_prices_aln)
print(f'✓ saved prices_aligned -> {out_prices_aln.resolve()}')

# === tính returns (linear & log) ===
ret_lin = prices_aligned.pct_change()
ret_log = np.log(prices_aligned / prices_aligned.shift(1))

ret_lin.to_parquet(out_ret_lin)
ret_log.to_parquet(out_ret_log)
print(f'✓ saved returns -> {out_ret_lin.name}, {out_ret_log.name}')

# === tách benchmark returns (SPY) ===
if 'SPY' not in ret_lin.columns:
    raise ValueError('SPY not found in columns. Check raw data.')
bench_ret = ret_lin[['SPY']].copy()
bench_ret.to_parquet(out_bench_ret)
print(f'✓ saved benchmark_ret -> {out_bench_ret.resolve()}')

print('\nSanity checks:')
print('ret_lin first valid date:', ret_lin.dropna(how="all").index.min())
print('Any NaN tail?', ret_lin.tail(5).isna().sum().sum() > 0)

print('\nNext: run 03_make_weights.py')

✓ saved prices_aligned -> C:\Users\ACER\python\Dissertation\Port Dashboard\data\processed\prices_aligned.parquet
✓ saved returns -> returns_linear.parquet, returns_log.parquet
✓ saved benchmark_ret -> C:\Users\ACER\python\Dissertation\Port Dashboard\data\processed\benchmark_ret.parquet

Sanity checks:
ret_lin first valid date: 2018-01-03 00:00:00
Any NaN tail? False

Next: run 03_make_weights.py


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

cal = pd.read_parquet("data/processed/calendar.parquet")
cal['date'] = pd.to_datetime(cal['date'])
cal = cal.set_index('date').index

pa  = pd.read_parquet("data/processed/prices_aligned.parquet")
rl  = pd.read_parquet("data/processed/returns_linear.parquet")
rlog= pd.read_parquet("data/processed/returns_log.parquet")
br  = pd.read_parquet("data/processed/benchmark_ret.parquet")

# calendar
assert cal.is_monotonic_increasing and cal.is_unique, "Calendar lỗi (không tăng dần hoặc duplicate)"
print("Calendar length:", len(cal))

# prices_aligned
assert pa.index.equals(cal), "prices_aligned phải cùng index với calendar"
assert (pa > 0).all().all(), "Có giá <= 0 trong prices_aligned"
print("prices_aligned shape:", pa.shape)

# returns_linear & log
assert rl.index.equals(cal) and rlog.index.equals(cal), "Returns index lệch calendar"
first_valid = rl.dropna(how="all").index.min()
print("First valid return date:", first_valid.date())

# NaN chỉ ở dòng đầu
rl_na = rl.iloc[1:].isna().sum().sum()
rlog_na = rlog.iloc[1:].isna().sum().sum()
assert rl_na == 0 and rlog_na == 0, "Có NaN ngoài dòng đầu tiên trong returns"

# Kiểm tra gần đúng: (1+lin) ≈ exp(log)
approx = np.allclose((1+rl.iloc[1:]).values, np.exp(rlog.iloc[1:]).values, rtol=1e-6, atol=1e-8)
assert approx, "Quan hệ (1+ret_lin) ≈ exp(ret_log) không khớp"

# benchmark_ret
assert 'SPY' in br.columns, "benchmark_ret phải có cột SPY"
assert br.index.equals(cal), "benchmark_ret index lệch calendar"

print("PROCESSED: OK ✅")

Calendar length: 1825
prices_aligned shape: (1825, 14)
First valid return date: 2018-01-03
PROCESSED: OK ✅
